##### 모델 로드

In [1]:
# transformers 모듈: 
# - Hugging Face에서 제공하는 라이브러리
# - 다양한 사전 학습된 모델과 토크나이저를 쉽게 사용할 수 있도록 지원
from transformers import AutoModel

# 사용할 모델 이름
model_name = "klue/bert-base"

# 모델 로드 (사전 학습된 가중치 포함)
# - 분류 헤드 없이 Encoder 출력만 반환 (임베딩 추출용)
model = AutoModel.from_pretrained(model_name)

# 모델 이름 출력
print(f"모델 이름: {model_name}", "\n")

# 모델 아키텍처
config = model.config
print(f"입력 최대 토큰 수: {config.max_position_embeddings}")
print(f"토큰 벡터 차원(d_model): {config.hidden_size}")
print(f"Encoder Block 수: {config.num_hidden_layers}")
print(f"MHA 헤드 수: {config.num_attention_heads}")
print(f"FFN 차원: {config.intermediate_size}") 
print(f"드롭아웃: {config.hidden_dropout_prob}", "\n")

# 파라미터 수 확인
total_params = sum(p.numel() for p in model.parameters())
print(f"전체 파라미터 수: {total_params:,}")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: klue/bert-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


모델 이름: klue/bert-base 

입력 최대 토큰 수: 512
토큰 벡터 차원(d_model): 768
Encoder Block 수: 12
MHA 헤드 수: 12
FFN 차원: 3072
드롭아웃: 0.1 

전체 파라미터 수: 110,617,344


##### 단일 문장 토큰화

In [17]:
# transformers 모듈: 
# - Hugging Face에서 제공하는 라이브러리
# - 다양한 사전 학습된 모델과 토크나이저를 쉽게 사용할 수 있도록 지원
from transformers import AutoTokenizer

# 사용할 모델 이름
model_name = "klue/bert-base"

# 토크나이저 로드
# - 모델과 동일한 이름으로 로드하여 호환되는 토크나이저 사용
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 입력 문장
text = "자연어 처리를 학습합니다."

# 토큰화하기
tokens = tokenizer.tokenize(text)
print(f"토큰 문자열: {tokens}", "\n")

# 토크나이저로 문장을 모델 입력 형태로 인코딩
inputs = tokenizer(
    text,                  # 입력 문장   
    max_length=32,         # 최대 입력 토큰 수 (BERT 논문 기본값은 512)
    truncation=True,       # 최대 입력 토큰수(max_length)를 초과하면 뒤를 잘라냄    
    return_tensors="pt",   # PyTorch 텐서로 반환 (tf는 TensorFlow)
)
print("토크나이저 출력 키 목록:", list(inputs.keys()), "\n")

# input_ids: 
# - 각 토큰을 정수 ID로 변환한 값 (맨 앞 [CLS]=2, 맨 뒤 [SEP]=3)
print(f"input_ids:      {inputs['input_ids']}")

# attention_mask: 
# - 실제 토큰과 패딩 토큰을 구분하는 마스크 (1=실제 토큰, 0=[PAD] 토큰)
# - 단일 문장 입력일 경우 모두 1
# - 배치 입력일 경우 최대 문장 길이로 다른 문장들의 길이를 맞추므로 [PAD] 토큰이 생김
print(f"attention_mask: {inputs['attention_mask']}")

# token_type_ids: 
# - 문장 구분 (문장A=0, 문장B=1)
# - 단일 문장이면 모두 0
print(f"token_type_ids: {inputs['token_type_ids']}", "\n")

# 토큰 ID → 토큰 문자열로 역변환해서 확인
# inputs['input_ids'] shape: (1, seq_len) → [0]으로 1D 텐서 추출
tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
print(f"토큰 문자열로 복원: {tokens}")

토큰 문자열: ['자연', '##어', '처리', '##를', '학습', '##합니다', '.'] 

토크나이저 출력 키 목록: ['input_ids', 'token_type_ids', 'attention_mask'] 

input_ids:      tensor([[    2,  3941,  2051,  4211,  2138,  4611, 11800,    18,     3]])
attention_mask: tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1]])
token_type_ids: tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0]]) 

토큰 문자열로 복원: ['[CLS]', '자연', '##어', '처리', '##를', '학습', '##합니다', '.', '[SEP]']


##### 두 개의 입력 문장 토큰화

In [4]:
# 입력 문장
text1 = "영화 재미있었어요."
text2 = "배우들이 훌륭해요."

# 토큰화하기
tokens = tokenizer.tokenize(text1, text2)  # 문장A, 문장B 입력
print(f"토큰화 결과: {tokens}", "\n")

# 토크나이저로 문장을 모델 입력 형태로 인코딩
inputs = tokenizer(
    text1, text2,          # 문장A, 문장B 입력  
    max_length=32,         # 최대 입력 토큰 수 (BERT 논문 기본값은 512)
    truncation=True,       # 최대 입력 토큰수(max_length)를 초과하면 뒤를 잘라냄    
    return_tensors="pt",   # PyTorch 텐서로 반환 (tf는 TensorFlow)
)
print("토크나이저 출력 키 목록:", list(inputs.keys()), "\n")

# input_ids: 
# - 각 토큰을 정수 ID로 변환한 값 (맨 앞 [CLS]=2, 맨 뒤 [SEP]=3)
print(f"input_ids: {inputs['input_ids']}")

# token_type_ids: 
# - 문장 구분 (문장A=0, 문장B=1)
print(f"token_type_ids: {inputs['token_type_ids']}")

# attention_mask: 
# - 실제 토큰과 패딩 토큰을 구분하는 마스크 (1=실제 토큰, 0=[PAD] 토큰)
# - 배치 입력일 경우 최대 문장 길이로 다른 문장들의 길이를 맞추므로 [PAD] 토큰이 생김
print(f"attention_mask: {inputs['attention_mask']}")
print()

# 토큰 ID → 토큰 문자열로 역변환해서 확인
# inputs['input_ids'] shape: (1, seq_len) → [0]으로 1D 텐서 추출
tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
print(f"토큰 문자열로 복원: {tokens}")

토큰화 결과: ['영화', '재미있', '##었', '##어요', '.', '배우', '##들이', '훌륭', '##해', '##요', '.'] 

토크나이저 출력 키 목록: ['input_ids', 'token_type_ids', 'attention_mask'] 

input_ids: tensor([[    2,  3771,  6001,  2359, 10283,    18,     3,  4165,  7285,  5825,
          2097,  2182,    18,     3]])
token_type_ids: tensor([[0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1]])
attention_mask: tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])

토큰 문자열로 복원: ['[CLS]', '영화', '재미있', '##었', '##어요', '.', '[SEP]', '배우', '##들이', '훌륭', '##해', '##요', '.', '[SEP]']


##### 배치 입력과 패딩

In [7]:
# 배치 입력 문장
sentences = [
    "이 영화 정말 재미있었어요!",
    "CG가 너무 많이 들어갔네요. 현실성이 없어요",
    "그냥 평범한 영화였습니다.",
]

# 토크나이저로 문장을 모델 입력 형태로 인코딩
inputs = tokenizer(
    sentences,
    max_length=32,    
    truncation=True,
    # 배치 내 가장 긴 문장 길이에 맞춰 짧은 문장에 [PAD]:0 토큰 추가
    padding=True,    
    return_tensors="pt",
)

print(f"input_ids:\n {inputs['input_ids']}\n")
print(f"token_type_ids:\n {inputs['token_type_ids']}\n")
print(f"attention_mask:\n {inputs['attention_mask']}\n\n")

print(f"input_ids 크기: {inputs['input_ids'].shape}")
print(f"- 배치 크기(문장 수): {inputs['input_ids'].shape[0]}")
print(f"- 시퀀스 길이(최대 문장 기준 패딩): {inputs['input_ids'].shape[1]}")

input_ids:
 tensor([[    2,  1504,  3771,  3944,  6001,  2359, 10283,     5,     3,     0,
             0,     0,     0,     0],
        [    2, 11969,  2116,  3760,  3732,  5435,  2203,  2182,    18,  4017,
         11604,  1415, 10283,     3],
        [    2,  4181,  7131,  2470,  3771,  2507,  2219,  3606,    18,     3,
             0,     0,     0,     0]])

token_type_ids:
 tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])

attention_mask:
 tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0]])


input_ids 크기: torch.Size([3, 14])
- 배치 크기(문장 수): 3
- 시퀀스 길이(최대 문장 기준 패딩): 14


##### BERT의 특수 토큰

In [6]:
# BERT의 특수 토큰
# - 특수 토큰은 학습 데이터가 아닌 모델 구조를 위해 예약된 토큰
print(f"[PAD]  ID: {tokenizer.pad_token_id:>4}  → 패딩 (모델이 무시)")
print(f"[UNK]  ID: {tokenizer.unk_token_id:>4}  → 어휘 사전에 없는 미등록 토큰")
print(f"[CLS]  ID: {tokenizer.cls_token_id:>4}  → 문장 시작")
print(f"[SEP]  ID: {tokenizer.sep_token_id:>4}  → 문장 구분 (두 문장 입력 시 사이에 삽입)")
print(f"[MASK] ID: {tokenizer.mask_token_id:>4}  → 사전학습 시 단어를 가리는 마스크 (MLM)")

[PAD]  ID:    0  → 패딩 (모델이 무시)
[UNK]  ID:    1  → 어휘 사전에 없는 미등록 토큰
[CLS]  ID:    2  → 문장 시작
[SEP]  ID:    3  → 문장 구분 (두 문장 입력 시 사이에 삽입)
[MASK] ID:    4  → 사전학습 시 단어를 가리는 마스크 (MLM)
